# VLM Compression Benchmark — Results Analysis

Reads all JSON results from `results/` and builds:
- Master comparison table
- Accuracy vs compression method plots
- Memory vs param count scatter
- Latency vs accuracy tradeoff curves
- Deployability flags for Raspberry Pi 5 (< 4 GB memory, < 10 s/sample)

In [ ]:
import json
import glob
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
RESULTS_ROOT = Path('../results')
print('Results root:', RESULTS_ROOT.resolve())

## 1. Load All Results

In [ ]:
def load_results(results_root: Path) -> pd.DataFrame:
    rows = []
    for method_dir in sorted(results_root.iterdir()):
        if not method_dir.is_dir():
            continue
        method = method_dir.name  # baseline / ptq / pruning / qlora
        for json_path in sorted(method_dir.glob('*.json')):
            with open(json_path) as f:
                data = json.load(f)

            base = {
                'method':       method,
                'model_id':     data.get('model_id', ''),
                'family':       data.get('family', ''),
                'params_M':     data.get('num_params_M', None),
                'quant':        data.get('quant', 'fp16'),
                'gpu_mem_load_mb': data.get('gpu_mem_load_mb', None),
                'compression_ratio': data.get('compression_ratio', None),
            }

            # Method-specific fields
            if method == 'pruning':
                base['sparsity'] = data.get('actual_sparsity', None)
                base['pruning_method'] = data.get('pruning_method', '')
            elif method == 'qlora':
                base['lora_rank'] = data.get('lora_rank', None)

            benchmarks = data.get('benchmarks', {})
            for bname, bdata in benchmarks.items():
                row = base.copy()
                row['benchmark']       = bname
                row['accuracy']        = bdata.get('accuracy', None)
                row['avg_latency_s']   = bdata.get('avg_latency_s', None)
                row['peak_memory_mb']  = bdata.get('peak_memory_mb', None)
                row['throughput_sps']  = bdata.get('throughput_sps', None)
                row['avg_power_w']     = bdata.get('avg_power_w', None)
                rows.append(row)

    return pd.DataFrame(rows)


df = load_results(RESULTS_ROOT)
print(f'Loaded {len(df)} rows from {df["model_id"].nunique()} model/config combos')
df.head()

## 2. Master Comparison Table

In [ ]:
DEPLOYABILITY_MEM_MB  = 4096   # < 4 GB
DEPLOYABILITY_LAT_S   = 10.0   # < 10 s/sample  (RPi5 target)

# Pivot: one row per model/method, columns = benchmark accuracies
pivot = df.pivot_table(
    index=['model_id', 'family', 'params_M', 'method', 'quant'],
    columns='benchmark',
    values=['accuracy', 'avg_latency_s', 'peak_memory_mb', 'throughput_sps'],
    aggfunc='mean',
).reset_index()

# Flatten multi-level column names
pivot.columns = [
    '_'.join(filter(None, col)).strip() if isinstance(col, tuple) else col
    for col in pivot.columns
]

# Deployability flag (based on VQAv2 if available)
mem_col = 'peak_memory_mb_vqav2' if 'peak_memory_mb_vqav2' in pivot.columns else None
lat_col = 'avg_latency_s_vqav2'  if 'avg_latency_s_vqav2'  in pivot.columns else None

if mem_col and lat_col:
    pivot['rpi5_deployable'] = (
        (pivot[mem_col] < DEPLOYABILITY_MEM_MB) &
        (pivot[lat_col] < DEPLOYABILITY_LAT_S)
    )

print(pivot.shape)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 60)
pivot

In [ ]:
# Export to CSV
pivot.to_csv('../results/master_table.csv', index=False)
print('Saved: results/master_table.csv')

## 3. Accuracy vs Compression Method per Family

In [ ]:
bench = 'vqav2'
plot_df = df[df['benchmark'] == bench].copy()

families = sorted(plot_df['family'].dropna().unique())
n_fam    = len(families)
ncols    = 2
nrows    = (n_fam + 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows), sharey=False)
axes = axes.flatten()

METHOD_ORDER = ['baseline', 'ptq', 'pruning', 'qlora']
PALETTE      = sns.color_palette('Set2', len(METHOD_ORDER))

for i, family in enumerate(families):
    ax    = axes[i]
    fdata = plot_df[plot_df['family'] == family]
    sns.barplot(
        data=fdata, x='method', y='accuracy', hue='quant',
        order=METHOD_ORDER, ax=ax, palette='tab10',
    )
    ax.set_title(family, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('VQAv2 Accuracy')
    ax.set_ylim(0, 1)
    ax.legend(title='quant', fontsize=8, title_fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('VQAv2 Accuracy per Family and Compression Method', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../results/accuracy_vs_method.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Memory vs Parameter Count Scatter

In [ ]:
bench    = 'vqav2'
plot_df  = df[(df['benchmark'] == bench)].dropna(subset=['params_M', 'peak_memory_mb'])

fig, ax = plt.subplots(figsize=(10, 6))

method_markers = {'baseline': 'o', 'ptq': 's', 'pruning': '^', 'qlora': 'D'}
palette = sns.color_palette('tab10', plot_df['family'].nunique())
family_colors = {f: palette[i] for i, f in enumerate(sorted(plot_df['family'].unique()))}

for _, row in plot_df.iterrows():
    ax.scatter(
        row['params_M'], row['peak_memory_mb'],
        marker=method_markers.get(row['method'], 'o'),
        color=family_colors.get(row['family'], 'gray'),
        s=80, alpha=0.75, edgecolors='white', linewidths=0.5,
    )

# RPi5 deployability zone
ax.axhline(DEPLOYABILITY_MEM_MB, color='red', linestyle='--', lw=1.2, label='RPi5 mem limit (4 GB)')

# Legends
from matplotlib.lines import Line2D
family_handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=8, label=f)
                  for f, c in family_colors.items()]
method_handles = [Line2D([0],[0], marker=m, color='gray', markersize=8, label=meth, linestyle='none')
                  for meth, m in method_markers.items()]

leg1 = ax.legend(handles=family_handles, title='Family',  loc='upper left',  fontsize=8)
leg2 = ax.legend(handles=method_handles, title='Method',  loc='upper right', fontsize=8)
ax.add_artist(leg1)
ax.add_artist(leg2)

ax.set_xlabel('Parameter Count (M)')
ax.set_ylabel('Peak GPU Memory (MB)')
ax.set_title('Memory vs Parameter Count by Family and Compression Method')
plt.tight_layout()
plt.savefig('../results/memory_vs_params.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Latency vs Accuracy Tradeoff Curves

In [ ]:
bench   = 'vqav2'
plot_df = df[(df['benchmark'] == bench)].dropna(subset=['avg_latency_s', 'accuracy'])

fig, ax = plt.subplots(figsize=(10, 6))

for family, grp in plot_df.groupby('family'):
    grp = grp.sort_values('avg_latency_s')
    ax.plot(grp['avg_latency_s'], grp['accuracy'], marker='o', label=family, linewidth=1.5)

ax.axvline(DEPLOYABILITY_LAT_S, color='red', linestyle='--', lw=1.2, label='RPi5 latency limit (10 s)')
ax.set_xlabel('Avg Latency per Sample (s)')
ax.set_ylabel('VQAv2 Accuracy')
ax.set_title('Latency vs Accuracy Tradeoff per Family')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../results/latency_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Deployability Summary (RPi5 Threshold)

In [ ]:
bench   = 'vqav2'
plot_df = df[(df['benchmark'] == bench)].dropna(subset=['avg_latency_s', 'peak_memory_mb'])

plot_df = plot_df.copy()
plot_df['rpi5_deployable'] = (
    (plot_df['peak_memory_mb'] < DEPLOYABILITY_MEM_MB) &
    (plot_df['avg_latency_s']  < DEPLOYABILITY_LAT_S)
)

deployable = plot_df[plot_df['rpi5_deployable']][['model_id','family','method','quant','accuracy','avg_latency_s','peak_memory_mb']]
print(f'Combinations meeting RPi5 deployability threshold: {len(deployable)}')
deployable.sort_values('accuracy', ascending=False)

## 7. Accuracy Drop: Baseline → Compression

In [ ]:
bench = 'vqav2'
bdf   = df[df['benchmark'] == bench][['model_id','method','quant','accuracy']].copy()

baseline_acc = bdf[bdf['method'] == 'baseline'].set_index('model_id')['accuracy'].to_dict()
bdf['baseline_acc'] = bdf['model_id'].map(baseline_acc)
bdf['acc_drop']     = bdf['baseline_acc'] - bdf['accuracy']

drop_df = bdf[bdf['method'] != 'baseline'].dropna(subset=['acc_drop'])

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=drop_df, x='model_id', y='acc_drop', hue='method', ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=7)
ax.set_xlabel('')
ax.set_ylabel('Accuracy Drop vs FP16 Baseline')
ax.set_title('VQAv2 Accuracy Drop per Model and Compression Method')
ax.axhline(0, color='black', lw=0.8)
ax.legend(title='Method', fontsize=8)
plt.tight_layout()
plt.savefig('../results/accuracy_drop.png', dpi=150, bbox_inches='tight')
plt.show()